# PRE: validated regeneration of all manuscript figures

This notebook orchestrates `pre_figures.py` and exports the six main figures, the supplementary schematic, and an old/new comparison PDF. It does **not** edit manuscript or rebuttal prose.

Original notebooks: `fig1_illustration.ipynb`; `damped_harmonic_oscillator/{sdho_zeta,theory_simulation_upcrossings,fano_scan_omega0,fano_scan_temp}.ipynb`; `OU_noise/OU_noise.ipynb`; `rational_quadratic/rational_quadratic.ipynb`. No source notebook was found for Fig. S1; its geometry is reconstructed from the stated change of variables.

The original quantitative notebooks call the now-corrected signed-erf formula classes. Their default fixed-grid integrator does not establish convergence, the temperature scan includes a degenerate zero-noise endpoint, and the OU export statements are commented out. Here the same physical parameter ranges are evaluated with stable conditional covariances, checked adaptive integration and controlled long-time tails. All valid Fig. 4 pixels are evaluated directly; no surface interpolation is used. Fig. 3 reuses the complete verified 50,000-trial archive.


In [ ]:
from pathlib import Path
import ast
import json
import os
import sys

here = Path.cwd().resolve()
CODE = next(p for p in (here, *here.parents) if (p / "gaussian_crossings").is_dir())
PAPER = CODE.parent / "upcrossing_theory_tex"
OUTPUT = CODE / "data/pre_figures_20260927"
FIGURES = PAPER / "pre_rebuttal/regenerated_figures"
os.environ["MPLCONFIGDIR"] = str(OUTPUT / "build/matplotlib")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(CODE))
sys.path.insert(0, str(CODE / "examples"))
import pre_figures as pre
print("Archive:", OUTPUT)
print("Figures:", FIGURES)


## Calculate, or verify and reuse this run's archive
For a fresh calculation, choose a new `OUTPUT` directory. Existing numerical archives are never overwritten. A cached run is accepted only if the archived source hash is intact and every numerical function matches the current source; plot-only changes are allowed. The independent validation is rerun when using cached arrays.

In [ ]:
if (OUTPUT / "arrays.npz").exists():
    run = json.loads((OUTPUT / "run.json").read_text())
    archived_source = OUTPUT / "run_source.py"
    assert pre.digest(archived_source) == run["source_sha256"]
    def numerical_definitions(path):
        tree = ast.parse(Path(path).read_text())
        excluded = {"plot", "style", "panel_letter", "fano_norm", "save", "comparison", "main"}
        return {node.name: ast.dump(node, include_attributes=False)
                for node in tree.body if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
                and node.name not in excluded}
    assert numerical_definitions(archived_source) == numerical_definitions(pre.__file__)
    if (OUTPUT / "render.json").exists():
        assert pre.digest(OUTPUT / "arrays.npz") == json.loads((OUTPUT / "render.json").read_text())["arrays_sha256"]
    print(json.dumps(pre.validate(), indent=2))
    print("Reusing verified arrays calculated during this regeneration pass.")
else:
    pre.calculate(OUTPUT)
print((OUTPUT / "validation.json").read_text())


In [ ]:
pre.plot(OUTPUT, FIGURES)
comparison_pdf = pre.comparison(OUTPUT, FIGURES, PAPER)
print((OUTPUT / "comparisons.json").read_text())
print("Comparison:", comparison_pdf)


## Interpretation and scope
- Mean crossing rates are unchanged analytically. Variance and Fano-factor figures change at nonzero thresholds.
- SDHO behavior depends on the normalized threshold as well as damping: even the plotted underdamped case exceeds one at sufficiently high levels.
- No reentrant return below one occurs on the checked OU heatmap. This is a statement about that parameter range, not a theorem for arbitrary Gaussian processes.
- Every displayed rational-quadratic curve has a resolved super-Poissonian maximum, including the squared-exponential limit. Tiny overshoots are quantified in the comparison table rather than inferred from line thickness.
- Revised Fig. 3 is unchanged numerically. Its already-disclosed two marginal confidence-interval misses remain present.
- The six PDFs and Fig. S1 are review assets in `pre_rebuttal/regenerated_figures`; the paper still requires a separate authorized prose/caption/rebuttal revision before submission.
